# HU das estruturas cardíacas no MM-WHS

Análise descritiva dos CTs de treino com labels de referência. Os sete rótulos são avaliados individualmente, em união e em relação às regiões não rotuladas. As intensidades são os HU originais do NIfTI, sem downscale ou threshold.

Cada exame tem o mesmo peso nos resultados gerais; o volume de uma estrutura não aumenta o peso daquele exame.

In [ ]:
# ruff: noqa: E402
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

REPO_ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").is_file()
)
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

from utils.project.datasets.ccta import discover_ccta_dataset
from utils.project.dataframe import require_series_column
from utils.project.analysis.mmwhs_heart_analysis import (
    COMBINED_REGION_NAMES,
    MMWHS_HEART_LABEL_SPECS,
    aggregate_mmwhs_heart_hu,
    load_mmwhs_heart_volume,
    select_mmwhs_heart_exams,
    summarize_mmwhs_heart_hu,
)
from utils.project.runtime.notebook_env import resolve_existing_path
from utils.visualization.images.intensity import plot_binned_intensity_histogram

## 1. Configuração e exames

`EXAM_IDS` substitui o sorteio quando recebe uma lista de IDs. Com `EXAM_IDS = None`, `N_EXAMS = None` usa todos os pares de treino disponíveis. A semente fixa permite repetir a mesma seleção.

In [ ]:
MMWHS_PATH = resolve_existing_path(
    "MMWHS_BASE_PATH",
    [Path("/run/media/matheus/HD/DatasetsCCTA/MM-WHS-2017-Dataset")],
    "MM-WHS 2017",
)
N_EXAMS: int | None = 10
RANDOM_SEED = 42
EXAM_IDS: list[str] | None = None
MARGIN_MM = 10.0
HU_BIN_EDGES = np.arange(-1500.0, 3001.0, 10.0)
DETAIL_EXAM_ID: str | None = None  # None mostra o primeiro exame selecionado.

inventory = discover_ccta_dataset("mmwhs", MMWHS_PATH)
selected_exams = select_mmwhs_heart_exams(
    inventory, n_exams=N_EXAMS, random_seed=RANDOM_SEED, exam_ids=EXAM_IDS
)
print(f"Exames selecionados: {len(selected_exams)}")
display(
    selected_exams.loc[
        :, ["exam_id", "slice_count", "spacing_x_mm", "spacing_y_mm", "spacing_z_mm"]
    ].round(3)
)

Os pares são canonicalizados e precisam ter shape e affine compatíveis. O carregamento aplica o escalonamento de intensidade do cabeçalho NIfTI. Um exame é liberado antes de carregar o seguinte; permanecem apenas estatísticas e histogramas.

**Fundo completo** contém todo voxel com label 0, inclusive ar e outros tecidos. **Fundo próximo ao coração** contém label 0 dentro da caixa delimitadora dos sete labels com margem de 10 mm. Essas regiões não equivalem a um tecido anatômico único.

In [ ]:
statistics_frames: list[pd.DataFrame] = []
histogram_frames: list[pd.DataFrame] = []
for _, record in selected_exams.iterrows():
    exam_id = str(record["exam_id"])
    image, label, spacing = load_mmwhs_heart_volume(
        str(record["path"]), str(record["label_path"])
    )
    exam_statistics, exam_histograms = summarize_mmwhs_heart_hu(
        image,
        label,
        spacing,
        exam_id=exam_id,
        margin_mm=MARGIN_MM,
        bin_edges=HU_BIN_EDGES,
    )
    statistics_frames.append(exam_statistics)
    histogram_frames.append(exam_histograms)
    print(f"{exam_id}: análise concluída")
    del image, label, exam_statistics, exam_histograms

statistics = pd.concat(statistics_frames, ignore_index=True)
histograms = pd.concat(histogram_frames, ignore_index=True)
summary, mean_densities = aggregate_mmwhs_heart_hu(statistics, histograms)
del statistics_frames, histogram_frames

## 2. Labels cardíacos

A tabela apresenta a **média das médias HU dos exames**, o desvio amostral entre essas médias e a **mediana das medianas HU dos exames**. Essas medidas descrevem os exames, não uma concatenação de voxels. O desvio fica vazio quando apenas um exame contribui.

Cada curva é a média das densidades individuais. Labels sem voxels finitos não contribuem. A coluna “Fora dos bins” informa a porcentagem média fora da faixa exibida; esses valores continuam nas estatísticas. O volume e a quantidade de voxels aparecem no detalhe por exame.

In [ ]:
SUMMARY_COLUMNS = {
    "region_name": "Região",
    "exam_count": "Exames válidos",
    "selected_exam_count": "Exames selecionados",
    "mean_of_means_hu": "Média das médias (HU)",
    "std_between_exam_means_hu": "DP entre médias (HU)",
    "median_of_medians_hu": "Mediana das medianas (HU)",
    "mean_outside_range_percent": "Fora dos bins (%)",
    "nonfinite_voxel_count": "Voxels não finitos",
}
label_regions = [f"label_{value}" for value in MMWHS_HEART_LABEL_SPECS]


def show_region_summary(regions: list[str]) -> None:
    frame = summary.loc[require_series_column(summary, "region").isin(regions)]
    display(
        frame.loc[:, list(SUMMARY_COLUMNS)].rename(columns=SUMMARY_COLUMNS).round(2)
    )


show_region_summary(label_regions)

In [ ]:
fig = plt.figure(figsize=(13, 14))
for index, (value, (name, color)) in enumerate(
    MMWHS_HEART_LABEL_SPECS.items(), start=1
):
    ax = fig.add_subplot(4, 2, index)
    region = f"label_{value}"
    row = summary.loc[require_series_column(summary, "region").eq(region)].iloc[0]
    if int(row["exam_count"]) == 0:
        ax.text(0.5, 0.5, "Sem voxels finitos", ha="center", transform=ax.transAxes)
    else:
        distribution = mean_densities.loc[
            require_series_column(mean_densities, "region").eq(region)
        ]
        plot_binned_intensity_histogram(
            ax,
            distribution,
            value_column="density",
            mean_hu=float(row["mean_of_means_hu"]),
            median_hu=float(row["median_of_medians_hu"]),
            distribution_label="Densidade média dos exames",
            color=f"#{color:06x}",
        )
        ax.legend(fontsize=8)
    ax.set(
        title=f"{name} ({value}) — {int(row['exam_count'])} exames",
        xlabel="Intensidade (HU)",
        ylabel="Densidade (1/HU)",
        xlim=(HU_BIN_EDGES[0], HU_BIN_EDGES[-1]),
    )
    ax.grid(alpha=0.2)
fig.tight_layout()
plt.show()

## 3. União cardíaca e fundo

A união usa todos os voxels dos sete labels de cada exame. Sua média não é a média simples das médias das estruturas: estruturas maiores contribuem com mais voxels **dentro do exame**, enquanto todos os exames mantêm peso igual no agregado.

As distribuições mostram como as intensidades rotuladas se relacionam com o fundo completo e o contexto próximo ao coração. A integração da curva na faixa exibida pode ser menor que 1 quando há valores fora dos bins.

In [ ]:
combined_regions = list(COMBINED_REGION_NAMES)
show_region_summary(combined_regions)

fig, ax = plt.subplots(figsize=(11, 4.5))
combined_colors = {
    "heart": "#e53935",
    "background_full": "#607d8b",
    "background_local": "#1e88e5",
}
for region, color in combined_colors.items():
    distribution = mean_densities.loc[
        require_series_column(mean_densities, "region").eq(region)
    ]
    ax.step(
        require_series_column(distribution, "bin_center_hu"),
        require_series_column(distribution, "density"),
        where="mid",
        color=color,
        linewidth=1.5,
        label=COMBINED_REGION_NAMES[region],
    )
ax.set(
    xlabel="Intensidade (HU)",
    ylabel="Densidade média dos exames (1/HU)",
    title="Coração completo e regiões não rotuladas",
    xlim=(HU_BIN_EDGES[0], HU_BIN_EDGES[-1]),
)
ax.legend()
ax.grid(alpha=0.2)
fig.tight_layout()
plt.show()

## 4. Resultados por exame

A primeira tabela mostra a média HU de cada região em cada exame. No detalhe, os percentis são calculados exatamente sobre todos os voxels finitos da região, sem amostragem. Labels ausentes têm volume zero e estatísticas vazias. O volume inclui todos os voxels da região; as intensidades não finitas são contadas separadamente.

In [ ]:
region_names = [name for name, _ in MMWHS_HEART_LABEL_SPECS.values()] + list(
    COMBINED_REGION_NAMES.values()
)
exam_means = statistics.pivot(index="exam_id", columns="region_name", values="mean_hu")
display(exam_means.reindex(columns=region_names).round(1))

detail_exam_id = DETAIL_EXAM_ID or str(selected_exams.iloc[0]["exam_id"])
if detail_exam_id not in set(require_series_column(selected_exams, "exam_id")):
    raise ValueError(f"DETAIL_EXAM_ID não pertence à seleção: {detail_exam_id}")
detail = statistics.loc[require_series_column(statistics, "exam_id").eq(detail_exam_id)]
DETAIL_COLUMNS = {
    "region_name": "Região",
    "voxel_count": "Voxels",
    "valid_voxel_count": "Voxels finitos",
    "volume_ml": "Volume (ml)",
    "mean_hu": "Média HU",
    "std_hu": "DP HU",
    "median_hu": "Mediana HU",
    "q1_hu": "Q1 HU",
    "q3_hu": "Q3 HU",
    "p5_hu": "P5 HU",
    "p95_hu": "P95 HU",
    "min_hu": "Mínimo HU",
    "max_hu": "Máximo HU",
    "below_range_count": "Abaixo dos bins",
    "above_range_count": "Acima dos bins",
    "nonfinite_voxel_count": "Não finitos",
}
print(f"Detalhe: {detail_exam_id}")
display(detail.loc[:, list(DETAIL_COLUMNS)].rename(columns=DETAIL_COLUMNS).round(2))